# Thành viên A – Bước 1 & 2: Kiểm tra 369 ca NIfTI 3D, EDA & Tiền xử lý BraTS 2020

Chạy **1 phiên Colab (CPU hoặc T4 + RAM cao cấp)** để thực hiện trọn gói:
1. Tải bộ BraTS 2020 NIfTI 3D gốc (`awsaf49/brats20-dataset-training-validation`) từ Kaggle.
2. Kiểm tra toàn vẹn 369 ca (`validate_nifti.py`).
3. Chạy EDA thống kê & vẽ 4 biểu đồ (`eda_brats2020.py`).
4. Chạy tiền xử lý theo Quy ước v1.1 (`preprocess_nifti.py`) → xuất 369 file `.npz` + `metadata/*.json`.
5. Lưu kết quả EDA vào Google Drive và đẩy bản dữ liệu sạch lên **Kaggle Private Dataset** (share cho B).

In [ ]:
# 1. Mount Google Drive để lưu kết quả EDA (CSV, 4 biểu đồ) & Clone repo GitHub
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/<USERNAME>/brats-tumor-seg.git'  # <-- Sửa tên nick GitHub của bạn
!git clone {REPO_URL}
%cd brats-tumor-seg
!pip install -q -r requirements.txt

In [ ]:
# 2. Cấu hình Kaggle API (upload file kaggle.json từ máy bạn)
from google.colab import files
import os

if not os.path.exists('/root/.kaggle/kaggle.json'):
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# 3. Tải bộ BraTS 2020 NIfTI 3D gốc từ Kaggle vào /content/brats2020_raw
!kaggle datasets download -d awsaf49/brats20-dataset-training-validation -p /content/brats2020_raw --unzip

In [ ]:
# 4. Kiểm tra toàn vẹn 369 ca (4 chuỗi + seg, shape/affine, lưới trực giao, nhãn {0,1,2,4})
DATA_DIR = '/content/brats2020_raw/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'
!python -m src.preprocess.validate_nifti --data-dir {DATA_DIR} --output-csv results/nifti_audit.csv

In [ ]:
# 5. Chạy EDA BraTS 2020: xuất CSV thống kê (gửi B) + 4 biểu đồ báo cáo
!python -m src.preprocess.eda_brats2020 --data-dir {DATA_DIR} --splits configs/splits.json --out-dir results

In [ ]:
# Hiển thị nhanh 4 biểu đồ EDA vừa tạo
from IPython.display import Image, display
for name in [
    'eda_01_tumor_volume_dist.png',
    'eda_02_intensity_distributions.png',
    'eda_03_class_imbalance_slices.png',
    'eda_04_sample_cases.png',
]:
    display(Image(filename=f'results/figures/{name}'))

In [ ]:
# 6. Tiền xử lý toàn bộ 369 ca theo Quy ước v1.1 -> xuất 369 file .npz + metadata/*.json
OUT_DIR = '/content/preprocessed_brats2020'
!python -m src.preprocess.preprocess_nifti --data-dir {DATA_DIR} --out-dir {OUT_DIR} --image-size 224
!du -sh {OUT_DIR}

In [ ]:
# 7. Copy kết quả EDA (CSV, audit, 4 biểu đồ) sang Google Drive để lưu trữ và tải về máy
DRIVE_RES = '/content/drive/MyDrive/brats_midterm_results'
!mkdir -p {DRIVE_RES}
!cp -r results/* {DRIVE_RES}/
print('Đã lưu toàn bộ CSV và biểu đồ vào:', DRIVE_RES)

In [ ]:
# 8. Đẩy bản dữ liệu đã tiền xử lý lên Kaggle Private Dataset (chỉ chạy 1 lần)
import json

KAGGLE_USERNAME = '<KAGGLE_USERNAME>'  # <-- Sửa thành username Kaggle trong file kaggle.json của bạn
DATASET_SLUG = 'brats2020-preprocessed-224-v11'

meta = {
    'title': 'BraTS 2020 Preprocessed 224x224 v1.1',
    'id': f'{KAGGLE_USERNAME}/{DATASET_SLUG}',
    'licenses': [{'name': 'CC0-1.0'}]
}
with open(f'{OUT_DIR}/dataset-metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

!kaggle datasets create -p {OUT_DIR} --dir-mode tar
print(f'Xong! Vào https://www.kaggle.com/datasets/{KAGGLE_USERNAME}/{DATASET_SLUG} để share cho thành viên B.')